## Testing different LLMs to test their performance on extracting information out of Presentation Slides



### Before getting started: extract each slide out of an example presentation
Example Data: [I3D:bio's Training Material for Omero](https://doi.org/10.5281/zenodo.8323588) (Schmidt, C., Bortolomeazzi, M. et al., 2023). For the following code the ['WhatIsOMERO.pdf'](https://zenodo.org/records/8323588/files/202310_GENERAL_OMERO_Material_01_WhatIsOMERO.pdf?download=1) gets downloaded and processed.

In [ ]:
import sys
import os

# Add the root directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
from pdf2image import convert_from_path
from IPython.display import display
from PIL import Image
from pdf_utilities import load_pdf, save_images, text_extraction, download_zenodo_pdf
from endpoints import prompt_chatgpt, prompt_llama_11b, prompt_llama_90b, prompt_phi, prompt_gpt_mini

# Use the Record ID for the WhatIsOMERO.pdf File
zenodo_record_id = 8323588
pdf = download_zenodo_pdf(zenodo_record_id, pdf_number=1)

In [ ]:
slides = load_pdf(pdf)
display(slides[0])

### Option to save those Images on disc:
- If you want to save the Images in their original size, just ignore the *new_width*  parameter, e.g. 

        save_images(".", slides)
       
- If you want them to be saved in another folder, just add this folder path instead of ".", e.g.

        Windows: save_images(r"C:\Users\username\Documents\Slides", slides)

        macOS: save_images("/Users/username/Documents/Slides", slides)

        Linux: save_images("/home/username/Documents/Slides", slides)

In [ ]:
save_images(".", pdf, new_width=400)

## Text extraction
It is also possible to **extract the original text** from the PDF slides. This might be useful for comparison or further processing later on. This code will extract the text and saves it as image-text pairs in a dictionary on the disc:

You have to install pdfplumber for this:
`pip install pdfplumber`

In [ ]:
text_extraction(pdf, slides)

#### Loading a dictionary
The dictionary can also be loaded from the disc again like this:

In [ ]:
import yaml

# Load the YAML file containing the image paths and corresponding text
with open("dict_slides_text.yml", "r") as yaml_file:
    slide_dict = yaml.safe_load(yaml_file)

You can now have a look at the pairs, e.g. the first image-text pair.

In [ ]:
# Function to display an image-text pair
def display_image_text_pair(slide_dict, pdf_name, slide_number):
    # Construct the image key based on the PDF name and slide number
    image_path = f"{pdf_name}_slide{slide_number}.png"
    
    # Check if the image path exists in the dictionary
    if image_path in slide_dict:
        # Open and display the image
        img = Image.open(image_path)
        display(img)
        
        # Display the corresponding text
        text = slide_dict[image_path]
        print("Slide Text:", text)
    else:
        print(f"Slide {slide_number} for PDF '{pdf_name}' not found in the dictionary.")

# Replace 'WhatIsOMERO' and 1 with the desired PDF name and slide number
display_image_text_pair(slide_dict, "WhatIsOMERO", 1)

# Comparing different Models
The code below uses some helper functions to prompt the models. The whole functions can be found at [endpoints.py](https://github.com/NFDI4BIOIMAGE/SlideInsight/blob/main/endpoints.py).

## 1. [OpenAi GPT-4o](https://github.com/marketplace/models/azure-openai/gpt-4o)

In [ ]:
prompt = "Give me a short and precise summary (1-3 whole sentences) of what is displayed or explained in this Slide."

for i in range(1,(len(slides)+1)):
    result = prompt_chatgpt(f"slide{i}.png", prompt)
    print(f"Slide {i} Summary:\n", result, "\n")

## 2. [Llama-3.2-11B-Vision-Instruct](https://github.com/marketplace/models/azureml-meta/Llama-3-2-11B-Vision-Instruct)

In [ ]:
for i in range(1,(len(slides)+1)):
    result = prompt_llama_11b(f"slide{i}.png", prompt)
    print(f"Slide {i} Summary:\n", result, "\n")

## 3. [Phi-3.5-vision instruct (128k)](https://github.com/marketplace/models/azureml/Phi-3-5-vision-instruct)

In [ ]:
for i in range(1,(len(slides)+1)):
    result = prompt_phi(f"slide{i}.png", prompt)
    print(f"Slide {i} Summary:\n", result, "\n")

## 4. [Llama-3.2-90B-Vision-Instruct](https://github.com/marketplace/models/azureml-meta/Llama-3-2-90B-Vision-Instruct)

In [ ]:
for i in range(1,(len(slides)+1)):
    result = prompt_llama_90b(f"slide{i}.png", prompt)
    print(f"Slide {i} Summary:\n", result, "\n")

## 5. [GPT-4o mini](https://github.com/marketplace/models/azure-openai/gpt-4o-mini)

In [ ]:
for i in range(1,(len(slides)+1)):
    result = prompt_gpt_mini(f"slide{i}.png", prompt)
    print(f"Slide {i} Summary:\n", result, "\n")